# EAIM-Net v5 — Full Evaluation Notebook

Each dataset cell is **completely self-contained** — extracts only its own zip, evaluates, then deletes extracted data to free disk space.

| Cell | Dataset | Zip extracted | Pairs |
|------|---------|--------------|-------|
| 1 | Setup | — | — |
| 0b | Quick Resume | — | — |
| 2 | Evaluation engine | — | — |
| 4a | LOL eval15 | LOL.zip only | 15 |
| 4b | RESIDE-ITS | RESIDE_SOTS.zip only | ≤200 |
| 4c | Rain100L | Rain100L.zip only | 100 |
| 4d | Rain100H | Rain100H.zip only | ≤100 |
| 4e | DID-MDN | DID-MDN.zip only | ≤200 |
| 4f | SD1 | sd1.zip only | ≤200 |
| 4g | WTT Synthetic | weather_time_data.zip only | ≤200 |
| 5 | Summary | — | all |


---
## Cell 1 — Setup
*Run once per session.*

In [ ]:
# ================================================================
# CELL 1 - SETUP  (run once per session)
# ================================================================
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
def pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
pip("lpips","scikit-image","ipywidgets","tqdm","pyyaml","matplotlib")

import os, glob, json, math, time, random, warnings, io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as TF
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as GS
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.notebook import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
import lpips as lpips_lib
import ipywidgets as WG
from IPython.display import display as _D, clear_output
warnings.filterwarnings("ignore")

# ================================================================
# PATHS  -  ONLY EDIT THESE TWO LINES
# ================================================================
COLAB_ROOT  = "/content/eaim_v5"
WEATHER_DIR = "/content/drive/MyDrive/Weather"
#   ^^ Upload LOL.zip, RESIDE_SOTS.zip, Rain100L.zip, Rain100H.zip,
#      DID-MDN.zip, SD1.zip, weather_time_data.zip to this folder.
#      Everything else is automatic.

DRIVE_ROOT    = "/content/drive/MyDrive/adaptive_enhancement_training"
# ================================================================

DRIVE_CKPTS   = f"{DRIVE_ROOT}/checkpoints_v5"
DRIVE_PRE     = f"{DRIVE_ROOT}/pretrained_filters"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results_v5"
DRIVE_LOGS    = f"{DRIVE_ROOT}/logs_v5"
DATA_LOCAL    = "/content/weather_time_data"
LOG_FILE      = f"{DRIVE_ROOT}/training_log_v5.txt"

# ── Step 1: Create ALL directories first ─────────────────────
os.makedirs(COLAB_ROOT, exist_ok=True)          # MUST be before os.chdir
for d in [DRIVE_CKPTS, DRIVE_PRE, DRIVE_RESULTS, DRIVE_LOGS]:
    os.makedirs(d, exist_ok=True)

# ── Step 2: Change to working directory ──────────────────────
os.chdir(COLAB_ROOT)
sys.path.insert(0, COLAB_ROOT)

# ── Step 3: Copy source .py files from Drive → COLAB_ROOT ────
# Upload all .py files from src_v5/ folder on Drive once.
# They auto-copy here every session.
DRIVE_SRC = f"{DRIVE_ROOT}/src_v5"
import shutil as _shutil

PY_FILES = [
    "afb_module.py", "ess_module.py", "epe_module.py", "complete_model.py",
    "config.py", "dataset.py", "dataset_real_pairs.py",
    "losses.py", "dataset_loader.py", "dataset_loader_lazy.py",
    "pretrain_filters.py", "assemble_and_finetune.py",
]

print(f"Copying source files from {DRIVE_SRC} ...")
_missing = []
for _f in PY_FILES:
    _src = os.path.join(DRIVE_SRC, _f)
    _dst = os.path.join(COLAB_ROOT, _f)
    if os.path.exists(_src):
        _shutil.copy2(_src, _dst)
        print(f"  OK    {_f}")
    elif os.path.exists(_dst):
        print(f"  CACHE {_f}  (already present)")
    else:
        _missing.append(_f)
        print(f"  MISS  {_f}")

if _missing:
    print(f"\nWARNING: {len(_missing)} files missing from {DRIVE_SRC}")
    print("Please upload the src_v5/ folder to your Drive and re-run Cell 1.")
else:
    print(f"\nAll {len(PY_FILES)} source files ready in {COLAB_ROOT}")

# ── Auto-extract all datasets from zip files ──────────────────
from dataset_loader import (load_all_datasets, load_all_datasets_multi_drive,
                             mount_second_drive, load_sd1_from_second_drive,
                             get_loader_kwargs, PATHS)

# ================================================================
# SMART LAZY LOADING - datasets extracted ONE AT A TIME
# ================================================================
# No datasets are extracted here at setup time.
# Each pre-training cell (3-7) extracts only what IT needs,
# then cleans up before the next cell runs.
#
# This means:
#   - LOL.zip    (~0.3 GB) extracted only during Cell 3
#   - RESIDE.zip (~0.8 GB) extracted only during Cell 4
#   - Rain zips  (~1.5 GB) extracted only during Cell 5
#   - SD1.zip    (~10 GB)  extracted only during Cell 6, then deleted
#   - WTT.zip    (~0.2 GB) extracted only during Cell 7
#
# Maximum disk usage at any time: ~10 GB (only during Cell 6)
# After Cell 6 finishes: SD1 is deleted, disk drops back to ~0 GB
#
# SD1 on a second Google account? See Cell 6 for the commented
# mount_second_drive() option - no Drive space needed on either account.
# ================================================================

# Just verify the Weather folder exists and list available zips
if os.path.isdir(WEATHER_DIR):
    zips = sorted([f for f in os.listdir(WEATHER_DIR) if f.lower().endswith(".zip")])
    print(f"Weather folder: {WEATHER_DIR}")
    print(f"Zips available ({len(zips)}):")
    for z in zips:
        mb = os.path.getsize(os.path.join(WEATHER_DIR, z))/1e6
        print(f"  {z:<35} {mb:>8.1f} MB")
else:
    print(f"WARNING: Weather folder not found: {WEATHER_DIR}")
    print("Create it on Drive and upload your zip files.")

# These will be set by each pre-training cell when it runs
LOL_ROOT = RESIDE_ROOT = RAIN100L_ROOT = RAIN100H_ROOT = DIDMDN_ROOT = SD1_ROOT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FILTER_NAMES  = ["Low-light","Dehazing","Rain Removal","Illum. Norm.","Glare Reduction"]
WEATHER_NAMES = ["Clear","Rain","Fog","Light Snow","Glare"]
TIME_NAMES    = ["Dawn","Day","Dusk","Night"]
ILLUM_NAMES   = ["Low","Medium","High"]
FILTER_COLORS = ["#2278CF","#1D9E75","#EF9F27","#7F77DD","#D85A30"]

# ── Shared losses ─────────────────────────────────────────────
def _gauss(s=11,sig=1.5):
    c=torch.arange(s,dtype=torch.float32)-s//2
    g=torch.exp(-c**2/(2*sig**2)); return (g/g.sum()).outer(g/g.sum())

def ssim_loss(p,g,ws=11):
    C1,C2=0.01**2,0.03**2; B,C,H,W=p.shape
    win=_gauss(ws).to(p.device).expand(C,1,ws,ws); pad=ws//2
    mu1=TF.conv2d(p,win,padding=pad,groups=C); mu2=TF.conv2d(g,win,padding=pad,groups=C)
    s1=TF.conv2d(p*p,win,padding=pad,groups=C)-mu1**2
    s2=TF.conv2d(g*g,win,padding=pad,groups=C)-mu2**2
    s12=TF.conv2d(p*g,win,padding=pad,groups=C)-mu1*mu2
    return 1-(((2*mu1*mu2+C1)*(2*s12+C2))/((mu1**2+mu2**2+C1)*(s1+s2+C2))).mean()

def base_loss(pred,gt,ls=0.3): return nn.L1Loss()(pred,gt)+ls*ssim_loss(pred,gt)

def text_mask(gt,ks=5,th=0.02):
    gray=0.299*gt[:,0:1]+0.587*gt[:,1:2]+0.114*gt[:,2:3]
    lk=torch.tensor([[0.,-1.,0.],[-1.,4.,-1.],[0.,-1.,0.]],device=gt.device).view(1,1,3,3)
    return (TF.max_pool2d(TF.conv2d(gray,lk,padding=1).abs(),ks,1,ks//2)>th).float()

def text_aware_loss(pred,gt):
    return base_loss(pred,gt)+2.0*((pred-gt).abs()*text_mask(gt).detach()).mean()

# ── Shared metrics ────────────────────────────────────────────
LPIPS_FN = lpips_lib.LPIPS(net="alex").to(DEVICE)

def metrics(pred_np, gt_np):
    p=psnr_fn(gt_np,pred_np,data_range=1.0)
    s=ssim_fn(gt_np,pred_np,data_range=1.0,channel_axis=2)
    pt=torch.from_numpy(pred_np).permute(2,0,1).unsqueeze(0).to(DEVICE)
    gt=torch.from_numpy(gt_np ).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): lp=LPIPS_FN(pt*2-1,gt*2-1).item()
    return {"psnr":p,"ssim":s,"lpips":lp}

# ── Checkpoint helpers ────────────────────────────────────────
def _valid(p):
    if not os.path.exists(p) or os.path.getsize(p)<1024: return False
    try:
        import zipfile
        with zipfile.ZipFile(p,"r") as z: return len(z.namelist())>0
    except: return False

def find_latest(d):
    for p in sorted(glob.glob(os.path.join(d,"checkpoint_epoch_*.pth")),reverse=True):
        if _valid(p): return p
    best=os.path.join(d,"best_model.pth")
    return best if _valid(best) else None

def _exists(p): return bool(p) and os.path.isdir(p) and len(os.listdir(p))>0

def cleanup_old(d,keep=5):
    for p in sorted(glob.glob(os.path.join(d,"checkpoint_epoch_*.pth")))[:-keep]:
        try: os.remove(p)
        except: pass

def save_filter(m,name):
    p=os.path.join(DRIVE_PRE,f"{name}.pth")
    torch.save(m.state_dict(),p); print(f"  saved: {p}")

def load_filter(m,name):
    p=os.path.join(DRIVE_PRE,f"{name}.pth")
    if os.path.exists(p):
        m.load_state_dict(torch.load(p,map_location=DEVICE)); print(f"  loaded: {p}"); return True
    return False

# ── Visualization helpers ─────────────────────────────────────
def show_image_grid(images, titles, ncols=4, figsize_per=3.2, suptitle=""):
    nrows=math.ceil(len(images)/ncols)
    fig,axes=plt.subplots(nrows,ncols,figsize=(ncols*figsize_per,nrows*figsize_per))
    axes=np.array(axes).reshape(-1)
    for i,(img,title) in enumerate(zip(images,titles)):
        axes[i].imshow(img); axes[i].axis("off"); axes[i].set_title(title,fontsize=8)
    for j in range(i+1,len(axes)): axes[j].axis("off")
    if suptitle: fig.suptitle(suptitle,fontsize=11,fontweight="bold")
    plt.tight_layout(); plt.show()

def before_after_grid(module, dataset, n=6, title=""):
    module.eval()
    idxs=random.sample(range(len(dataset)),min(n,len(dataset)))
    imgs,titles=[],[]
    with torch.no_grad():
        for i in idxs:
            batch=dataset[i]
            inp=batch["input"].unsqueeze(0).to(DEVICE)
            tgt=batch["target"].unsqueeze(0).to(DEVICE)
            st=torch.ones(1,1).to(DEVICE)
            if "glare_map" in batch:
                pred=torch.clamp(module(inp,st,batch["glare_map"].unsqueeze(0).to(DEVICE)),0,1)
            else:
                pred=torch.clamp(module(inp,st),0,1)
            inp_np=inp[0].cpu().numpy().transpose(1,2,0)
            tgt_np=tgt[0].cpu().numpy().transpose(1,2,0)
            pred_np=pred[0].cpu().numpy().transpose(1,2,0)
            psnr_in=psnr_fn(tgt_np,inp_np,data_range=1.0)
            psnr_out=psnr_fn(tgt_np,pred_np,data_range=1.0)
            imgs+=[np.clip(inp_np,0,1),np.clip(pred_np,0,1),np.clip(tgt_np,0,1)]
            titles+=[f"Input\n{psnr_in:.1f}dB",f"Enhanced\n{psnr_out:.1f}dB","GT"]
    show_image_grid(imgs,titles,ncols=6,suptitle=title)

# ── Dataset helpers ───────────────────────────────────────────
def _to_sq(pil,sz): return pil.resize((sz,sz),Image.LANCZOS)

def _aug(inp,gt,sz):
    w,h=inp.size
    if w>sz and h>sz:
        x=random.randint(0,w-sz); y=random.randint(0,h-sz)
        inp=inp.crop((x,y,x+sz,y+sz)); gt=gt.crop((x,y,x+sz,y+sz))
    else: inp=_to_sq(inp,sz); gt=_to_sq(gt,sz)
    if random.random()>0.5:
        inp=inp.transpose(Image.FLIP_LEFT_RIGHT); gt=gt.transpose(Image.FLIP_LEFT_RIGHT)
    return inp,gt

TT=transforms.ToTensor()

def _find(root,inp_names,tgt_names,split=None):
    bases=[root]
    if split: bases=[os.path.join(root,split)]+bases
    for base in bases:
        if not os.path.isdir(base): continue
        subs={d.lower():d for d in os.listdir(base) if os.path.isdir(os.path.join(base,d))}
        i=next((os.path.join(base,subs[k]) for k in inp_names if k in subs),None)
        t=next((os.path.join(base,subs[k]) for k in tgt_names if k in subs),None)
        if i and t: return i,t
    raise FileNotFoundError(f"dirs not found under {root}")

# ── Dataset classes ───────────────────────────────────────────
class PairDS(Dataset):
    def __init__(self,inp_dir,tgt_dir,sz=256,augment=False,label=""):
        self.pairs=[]; self.sz=sz; self.aug=augment
        exts=(".jpg",".jpeg",".png")
        for f in sorted(os.listdir(inp_dir)):
            if not f.lower().endswith(exts): continue
            stem=os.path.splitext(f)[0]
            gt_stem=stem.split("_")[0] if "_" in stem else stem
            gt=None
            for gs in [stem,gt_stem]:
                for ext in exts:
                    c=os.path.join(tgt_dir,gs+ext)
                    if os.path.exists(c): gt=c; break
                if gt: break
            if gt: self.pairs.append((os.path.join(inp_dir,f),gt))
        print(f"  {label}: {len(self.pairs)} pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        ip,gp=self.pairs[idx]
        inp=Image.open(ip).convert("RGB"); gt=Image.open(gp).convert("RGB")
        if self.aug: inp,gt=_aug(inp,gt,self.sz)
        else: inp=_to_sq(inp,self.sz); gt=_to_sq(gt,self.sz)
        return {"input":TT(inp),"target":TT(gt),"name":os.path.basename(ip)}

class SD1DS(Dataset):
    def __init__(self,root,sz=256,augment=False):
        self.samples=[]; self.sz=sz; self.aug=augment
        for f in sorted(os.listdir(root)):
            if f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
                self.samples.append(os.path.join(root,f))
        print(f"  SD1 strip: {len(self.samples)} images")
    def __len__(self): return len(self.samples)
    def __getitem__(self,idx):
        img=Image.open(self.samples[idx]).convert("RGB"); w,h=img.size; pw=w//3
        clean=_to_sq(img.crop((0,0,pw,h)),self.sz)
        glare=_to_sq(img.crop((pw,0,pw*2,h)),self.sz)
        gmap =_to_sq(img.crop((pw*2,0,w,h)),self.sz)
        if self.aug and random.random()>0.5:
            clean=clean.transpose(Image.FLIP_LEFT_RIGHT)
            glare=glare.transpose(Image.FLIP_LEFT_RIGHT)
            gmap =gmap.transpose(Image.FLIP_LEFT_RIGHT)
        return {"input":TT(glare),"target":TT(clean),"glare_map":TT(gmap.convert("L")),"name":os.path.basename(self.samples[idx])}

class SD1TestDS(Dataset):
    def __init__(self,root,sz=256):
        self.pairs=[]; self.sz=sz; exts=(".jpg",".jpeg",".png")
        gt_d=os.path.join(root,"gt"); lt_d=os.path.join(root,"light")
        if os.path.isdir(gt_d) and os.path.isdir(lt_d):
            for f in sorted(os.listdir(gt_d)):
                if f.lower().endswith(exts):
                    lf=os.path.join(lt_d,f)
                    if os.path.exists(lf): self.pairs.append((lf,os.path.join(gt_d,f)))
        else:
            for gf in sorted(glob.glob(os.path.join(root,"*_gt.*"))):
                stem=os.path.basename(gf).split("_gt")[0]
                for ext in exts:
                    lf=os.path.join(root,stem+"_light"+ext)
                    if os.path.exists(lf): self.pairs.append((lf,gf)); break
        print(f"  SD1 test: {len(self.pairs)} pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        lp,gp=self.pairs[idx]
        inp=Image.open(lp).convert("RGB").resize((self.sz,self.sz),Image.LANCZOS)
        gt =Image.open(gp).convert("RGB").resize((self.sz,self.sz),Image.LANCZOS)
        return {"input":TT(inp),"target":TT(gt),"name":os.path.basename(lp)}

class WTTDS(Dataset):
    def __init__(self,root,split="train",sz=256):
        self.pairs=[]; self.sz=sz; self.aug=(split=="train")
        sd=os.path.join(root,split)
        id_=os.path.join(sd,"images"); cd=os.path.join(sd,"clean_images")
        if not os.path.isdir(id_): return
        for f in sorted(os.listdir(id_)):
            if not f.lower().endswith((".jpg",".jpeg",".png")): continue
            cp=os.path.join(cd,f)
            if os.path.exists(cp): self.pairs.append((os.path.join(id_,f),cp))
        print(f"  WTT {split}: {len(self.pairs)} pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        ip,gp=self.pairs[idx]
        inp=Image.open(ip).convert("RGB"); gt=Image.open(gp).convert("RGB")
        if self.aug: inp,gt=_aug(inp,gt,self.sz)
        else: inp=_to_sq(inp,self.sz); gt=_to_sq(gt,self.sz)
        return {"input":TT(inp),"target":TT(gt)}

# ── Filter trainer with live visualization ─────────────────────
def train_filter_viz(module,tr_ld,va_ld,name,n_epochs=25,lr=2e-4,loss_fn=None,color="#2278CF"):
    module=module.to(DEVICE)

    # ── Resume: load best weights + read log to find completed epochs ────
    best_psnr=0.0
    start_ep=0
    log_path=os.path.join(DRIVE_PRE,f"{name}_log.txt")

    if load_filter(module,f"{name}_best"):
        # Count how many epochs already done from log file
        if os.path.exists(log_path):
            with open(log_path) as _lf:
                completed=[l for l in _lf.readlines() if l.strip().startswith("Ep")]
                start_ep=len(completed)
                # Read best PSNR from log
                psnr_vals=[float(l.split("psnr=")[1].split("dB")[0])
                           for l in completed if "psnr=" in l]
                if psnr_vals: best_psnr=max(psnr_vals)
        if start_ep>=n_epochs:
            print(f"  [{name}] Already completed {start_ep}/{n_epochs} epochs.")
            print(f"  Best PSNR={best_psnr:.4f} dB. Skipping training.")
            return module
        print(f"  [{name}] Resuming from epoch {start_ep}/{n_epochs}  Best PSNR so far={best_psnr:.4f}")

    remaining=n_epochs-start_ep
    opt=optim.Adam(module.parameters(),lr=lr,weight_decay=1e-5)
    sch=optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=10,T_mult=1,eta_min=1e-6)
    # Fast-forward scheduler to match current epoch
    for _fep in range(start_ep): sch.step(_fep+1)

    ep_list=[]; loss_list=[]; psnr_list=[]; lines=[]
    out_curve=WG.Output(); out_status=WG.Output()
    _D(WG.VBox([WG.HTML(f"<h3 style='color:#1B3A6B'>Pre-training: {name} (ep {start_ep}→{n_epochs})</h3>"),out_status,out_curve]))
    for ep in range(start_ep, n_epochs):
        module.train(); tot=0.0
        for batch in tqdm(tr_ld,desc=f"{name} ep{ep:02d}",leave=False):
            inp=batch["input"].to(DEVICE); tgt=batch["target"].to(DEVICE)
            st=torch.ones(inp.shape[0],1).to(DEVICE)
            pred=module(inp,st,batch["glare_map"].to(DEVICE)) if name=="glare" and "glare_map" in batch else module(inp,st)
            loss=(loss_fn(pred,tgt,batch) if loss_fn else base_loss(pred,tgt))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(module.parameters(),1.0)
            opt.step(); tot+=loss.item()
        sch.step(ep+1)
        module.eval(); psnrs,ssims=[],[]
        with torch.no_grad():
            for batch in va_ld:
                inp=batch["input"].to(DEVICE); tgt=batch["target"].to(DEVICE)
                st=torch.ones(inp.shape[0],1).to(DEVICE)
                pred=torch.clamp(module(inp,st),0,1)
                for i in range(inp.shape[0]):
                    pn=np.clip(pred[i].cpu().numpy().transpose(1,2,0),0,1)
                    gn=np.clip(tgt[i].cpu().numpy().transpose(1,2,0),0,1)
                    psnrs.append(psnr_fn(gn,pn,data_range=1.0))
                    ssims.append(ssim_fn(gn,pn,data_range=1.0,channel_axis=2))
        vp=float(np.mean(psnrs)); vs=float(np.mean(ssims))
        lr_now=opt.param_groups[0]["lr"]; avg=tot/len(tr_ld)
        ep_list.append(ep); loss_list.append(avg); psnr_list.append(vp)
        line=f"Ep{ep:02d}  loss={avg:.4f}  psnr={vp:.3f}dB  ssim={vs:.4f}  lr={lr_now:.2e}"
        lines.append(line)
        with open(log_path,"a") as _lf: _lf.write(line+"\n")
        if vp > best_psnr + 0.005:
            best_psnr=vp
            save_filter(module, f"{name}_best")
        with out_status:
            clear_output(wait=True)
            print(f"  Epoch {ep+1}/{n_epochs}  Loss={avg:.4f}  PSNR={vp:.3f}dB  SSIM={vs:.4f}")
            print(f"  Best PSNR: {best_psnr:.4f} dB")
        if len(ep_list)>=2 and (ep%2==0 or ep==n_epochs-1):
            with out_curve:
                clear_output(wait=True)
                fig,(a1,a2)=plt.subplots(1,2,figsize=(12,3.5))
                a1.plot(ep_list,loss_list,color=color,lw=2,marker="o",ms=4)
                a1.set_title("Training Loss"); a1.set_xlabel("Epoch"); a1.grid(True,alpha=0.3)
                a2.plot(ep_list,psnr_list,color="#E8541A",lw=2,marker="s",ms=4)
                a2.set_title("Val PSNR (dB)"); a2.set_xlabel("Epoch"); a2.grid(True,alpha=0.3)
                fig.suptitle(f"{name} Pre-training",fontsize=11,fontweight="bold")
                plt.tight_layout(); plt.show()
    # Log already written per-epoch above
    print(f"\n  DONE [{name}]  Best PSNR={best_psnr:.4f} dB")
    return module

# ── Checkpoint info ───────────────────────────────────────────
gpu=torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "CPU"
print(f"Device : {DEVICE}  {gpu}")
print(f"\ncheckpoints_v5:")
for p in sorted(glob.glob(os.path.join(DRIVE_CKPTS,"*.pth"))):
    print(f"  {os.path.basename(p):<45} {os.path.getsize(p)/1e6:>6.1f} MB  {'OK' if _valid(p) else 'CORRUPT'}")
print("\nSetup complete. Datasets extracted above. Run Cell 2 next.")


# ── Extra imports for evaluation ─────────────────────────────
import matplotlib as _mpl
_mpl.rcParams.update({
    "font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],
    "font.size":11,"axes.titlesize":11,"axes.labelsize":11,
    "xtick.labelsize":10,"ytick.labelsize":10,"savefig.dpi":400,
})
SAVE_DPI = 400
EVAL_OUT = os.path.join(DRIVE_ROOT, "evaluation_results")
os.makedirs(EVAL_OUT, exist_ok=True)

import lpips as lpips_lib
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
from tqdm.notebook import tqdm

LPIPS_FN      = lpips_lib.LPIPS(net="alex").to(DEVICE)
FILTER_NAMES  = ["Low-light","Dehazing","Rain removal","Illum. norm","Glare red."]
FILTER_COLORS = ["#378ADD","#1D9E75","#EF9F27","#7F77DD","#D85A30"]
RESULTS       = {}

print(f"Eval output: {EVAL_OUT}")
print("Setup complete.")


---
## Cell 0b — Quick Resume
*Run after Cell 1 on reconnect.*

In [ ]:
# ================================================================
# CELL 0b — QUICK RESUME AFTER DISCONNECT
# Run this instead of Cells 2-7 when reconnecting mid-training.
# It restores all variables needed to continue from where you stopped.
#
# USAGE:
#   1. Run Cell 1 (Setup) — always required after disconnect
#   2. Run THIS cell — restores paths and variables
#   3. Jump directly to whichever cell you were on (3b/4/4b/5/6/7)
# ================================================================

print("Restoring session variables after disconnect...")

# ── Restore dataset paths from already-extracted folders ─────────
from dataset_loader_lazy import LOCAL_BASE, _EXTRACTED
import os

# Check what is already extracted on local disk
_datasets = {
    "LOL":        "/content/datasets/LOL",
    "RESIDE_SOTS":"/content/datasets/RESIDE_SOTS",
    "Rain100L":   "/content/datasets/Rain100L",
    "Rain100H":   "/content/datasets/Rain100H",
    "DID-MDN":    "/content/datasets/DID-MDN",
    "SD1":        "/content/datasets/SD1",
    "WTT":        "/content/datasets/WTT",
}

print("\nLocal disk status:")
for name, path in _datasets.items():
    if os.path.isdir(path):
        n = sum(1 for _,_,fs in os.walk(path) for f in fs
                if f.lower().endswith((".jpg",".png",".jpeg")))
        print(f"  OK    {name:<14} {n:>6} images  →  {path}")
    else:
        print(f"  MISS  {name:<14} (not on local disk — will re-extract if needed)")

# ── Restore convenience variables ─────────────────────────────────
from dataset_loader_lazy import _resolve_lol, _resolve_reside, _resolve_rain, _resolve_didmdn

def _get(path, resolver=None):
    if not os.path.isdir(path): return None
    return resolver(path) if resolver else path

LOL_ROOT      = _get("/content/datasets/LOL",        _resolve_lol)
RESIDE_ROOT   = _get("/content/datasets/RESIDE_SOTS", _resolve_reside)
RAIN100L_ROOT = _get("/content/datasets/Rain100L",   _resolve_rain)
RAIN100H_ROOT = _get("/content/datasets/Rain100H",   _resolve_rain)
DIDMDN_ROOT   = _get("/content/datasets/DID-MDN",    _resolve_didmdn)
SD1_ROOT      = _get("/content/datasets/SD1")
DATA_LOCAL    = "/content/datasets/WTT" if os.path.isdir("/content/datasets/WTT") else DATA_LOCAL

print("\nRestored path variables:")
for name, val in [("LOL_ROOT",LOL_ROOT),("RESIDE_ROOT",RESIDE_ROOT),
                   ("RAIN100L_ROOT",RAIN100L_ROOT),("RAIN100H_ROOT",RAIN100H_ROOT),
                   ("DIDMDN_ROOT",DIDMDN_ROOT),("SD1_ROOT",SD1_ROOT)]:
    status = "OK" if val and os.path.isdir(val) else "MISS"
    print(f"  {status}  {name} = {val}")

# ── Show pretrained filters status ────────────────────────────────
print("\nPretrained filters saved on Drive:")
import glob
pths = sorted(glob.glob(os.path.join(DRIVE_PRE, "*_best.pth")))
if pths:
    for p in pths:
        mb = os.path.getsize(p)/1e6
        print(f"  {os.path.basename(p):<35} {mb:>6.1f} MB")
else:
    print("  (none yet)")

# ── Glare-specific: show where training left off ───────────────────
glare_best = os.path.join(DRIVE_PRE, "glare_best.pth")
if os.path.exists(glare_best):
    import torch
    state = torch.load(glare_best, map_location="cpu")
    n_params = sum(v.numel() for v in state.values())
    print(f"\nglare_best.pth  →  {n_params:,} params loaded")
    print("Cell 6 will resume from this checkpoint automatically.")
    print("→ Re-run Cell 6 now (it calls load_filter at the top)")
else:
    print("\nNo glare_best.pth found — Cell 6 will start from scratch.")

print("\nResume complete. Jump to your target cell now.")
print()
print("IMPORTANT — After reconnect, datasets are NOT on local disk.")
print("Each cell auto-extracts what it needs from WEATHER_DIR zips.")
print()
print("For fine-tuning (Cell 9):")
print("  Cell 9 will auto-extract LOL + WTT + Rain100L before training.")
print("  Just run Cell 1 → Cell 0b → Cell 9.")
print()
print("For any pre-training cell (3-7):")
print("  Each cell extracts its own zip automatically.")
print("  Just run Cell 1 → Cell 0b → target cell.")


---
## Cell 2 — Evaluation Engine
*Run once. Defines all functions.*

In [ ]:
def _extract_one_zip(zip_name_patterns, extract_to, expected_subdir=None):
    """
    Extract a SINGLE zip file. Does NOT extract other zips.
    zip_name_patterns: list of glob patterns to find the zip, e.g. ["LOL*.zip","lol*.zip"]
    extract_to: destination folder e.g. /content/datasets
    expected_subdir: folder name inside zip to confirm success
    Returns path to extracted root or None.
    """
    import glob as _gl, zipfile as _zf
    os.makedirs(extract_to, exist_ok=True)

    # Find the zip file
    _found = []
    for _pat in zip_name_patterns:
        _found += _gl.glob(os.path.join(WEATHER_DIR, _pat))
    _found = list(dict.fromkeys(_found))  # deduplicate

    if not _found:
        print(f"  Zip not found. Patterns tried: {zip_name_patterns}")
        print(f"  Files in {WEATHER_DIR}:")
        for _f in sorted(os.listdir(WEATHER_DIR)):
            print(f"    {_f}")
        return None

    _zip_path = _found[0]
    _mb = os.path.getsize(_zip_path)/1e6
    print(f"  Found: {os.path.basename(_zip_path)}  ({_mb:.1f} MB)")

    # Check if already extracted
    if expected_subdir:
        _already = os.path.join(extract_to, expected_subdir)
        if os.path.isdir(_already):
            print(f"  Already extracted: {_already}")
            return _already

    # Extract
    print(f"  Extracting to {extract_to} ...")
    with _zf.ZipFile(_zip_path, "r") as _z:
        _z.extractall(extract_to)
    print(f"  Done.")

    # Find root
    if expected_subdir:
        _root = os.path.join(extract_to, expected_subdir)
        if os.path.isdir(_root):
            return _root

    # Auto-detect root (first new directory)
    _dirs = [d for d in os.listdir(extract_to)
             if os.path.isdir(os.path.join(extract_to,d))]
    if _dirs:
        return os.path.join(extract_to, _dirs[0])
    return extract_to


# ================================================================
# CELL 2 — EVALUATION ENGINE
# 400 dpi, Times New Roman, PNG + JPG — Word-ready figures
# Errors printed per image — no silent failures
# ================================================================

# ── Load best model (runs automatically when Cell 2 executes) ────
import glob as _gl2
print("Loading EAIM-Net model...")
from complete_model import AdaptiveEnhancementModel
from config import CONFIG

def _find_best_checkpoint(ckpts_dir):
    """Find best_model.pth, else latest epoch checkpoint."""
    _best = os.path.join(ckpts_dir, "best_model.pth")
    if os.path.exists(_best):
        return _best
    _eps = sorted(_gl2.glob(os.path.join(ckpts_dir,"checkpoint_epoch_*.pth")))
    return _eps[-1] if _eps else None

_ckpt_path = _find_best_checkpoint(DRIVE_CKPTS)
assert _ckpt_path and os.path.exists(_ckpt_path), (
    f"No checkpoint found in {DRIVE_CKPTS}\n"
    "Run Cell 1 first to set DRIVE_CKPTS."
)

MODEL = AdaptiveEnhancementModel(
    backbone            = CONFIG["model"]["backbone"],
    num_weather_classes = CONFIG["model"]["num_weather_classes"],
    num_time_classes    = CONFIG["model"]["num_time_classes"],
    num_illum_classes   = CONFIG["model"]["num_illum_classes"],
    feature_dim         = CONFIG["model"]["feature_dim"],
    num_filters         = CONFIG["model"]["num_filters"],
    pretrained          = False,
).to(DEVICE)

_ck = torch.load(_ckpt_path, map_location=DEVICE)
try:    MODEL.load_state_dict(_ck["model_state_dict"])
except: MODEL.load_state_dict(_ck["model_state_dict"], strict=False)
MODEL.eval()

_sz = os.path.getsize(_ckpt_path)/1e6
print(f"  Loaded : {os.path.basename(_ckpt_path)}  ({_sz:.1f} MB)")
print(f"  Device : {DEVICE}")
print(f"  Params : {sum(p.numel() for p in MODEL.parameters()):,}")
print()


def _run_model(model, inp_pil):
    IW, IH = inp_pil.size
    PW = ((IW+3)//4)*4
    PH = ((IH+3)//4)*4
    pad = Image.new("RGB", (PW, PH))
    pad.paste(inp_pil, (0, 0))
    t = transforms.ToTensor()(pad).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(t)
    enh = out["enhanced"][0].cpu().numpy().transpose(1,2,0)[:IH,:IW]
    fw  = out["filter_weights"][0].cpu().numpy()
    H   = out["weight_entropy"][0].item()
    wl  = out["weather_logits"][0].argmax().item()
    return np.clip(enh, 0, 1), fw, H, wl


def _metrics(pred, gt):
    p  = psnr_fn(gt, pred, data_range=1.0)
    s  = ssim_fn(gt, pred, data_range=1.0, channel_axis=2)
    pt   = torch.from_numpy(pred.copy()).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
    gt_t = torch.from_numpy(gt.copy()  ).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        lp = LPIPS_FN(pt*2-1, gt_t*2-1).item()
    return p, s, lp


def _resize_match(inp_np, gt_np):
    if inp_np.shape == gt_np.shape:
        return gt_np
    ih, iw = inp_np.shape[:2]
    gt_pil = Image.fromarray((gt_np*255).astype(np.uint8))
    gt_pil = gt_pil.resize((iw, ih), Image.LANCZOS)
    return np.array(gt_pil).astype(np.float32) / 255.0


def make_figure(label, samples, result):
    """Times New Roman enforced inside rc_context so other cells cannot override."""
    import matplotlib
    _ctx = {
        "font.family":"serif",
        "font.serif":["Times New Roman","DejaVu Serif"],
        "font.size":11,"axes.titlesize":11,"axes.labelsize":11,
        "xtick.labelsize":10,"ytick.labelsize":10,
        "savefig.dpi":400,
    }
    with matplotlib.rc_context(_ctx):
        _make_figure_inner(label, samples, result)

def _make_figure_inner(label, samples, result):
    n = len(samples)
    if n == 0:
        print(f"  No samples for {label}")
        return
    col_w = 2.5
    fig_w = n * col_w + 0.9
    fig_h = fig_w * 0.86
    fig = plt.figure(figsize=(fig_w, fig_h), facecolor="white")
    gs  = GS.GridSpec(3, 1, figure=fig, height_ratios=[4,4,2.0],
                      hspace=0.14, left=0.09, right=0.97, top=0.93, bottom=0.01)
    gs_top = GS.GridSpecFromSubplotSpec(1, n, subplot_spec=gs[0], wspace=0.04)
    gs_bot = GS.GridSpecFromSubplotSpec(1, n, subplot_spec=gs[1], wspace=0.04)

    for k, s in enumerate(samples):
        dp  = s["pe"] - s["pi"]
        col = "#1a6b1a" if dp >= 0 else "#b31b1b"

        ax_i = fig.add_subplot(gs_top[0, k])
        ax_i.imshow(np.clip(s["inp_np"], 0, 1), interpolation="lanczos")
        ax_i.axis("off")
        ax_i.set_title(f"PSNR {s['pi']:.2f} dB,  SSIM {s['si']:.4f}",
                       fontsize=9, pad=4, color="#222")
        if k == 0:
            ax_i.text(-0.08, 0.5, "Input", transform=ax_i.transAxes,
                      ha="right", va="center", rotation=90,
                      fontsize=11, fontweight="bold", color="#1B3A6B")

        ax_e = fig.add_subplot(gs_bot[0, k])
        ax_e.imshow(np.clip(s["enh_np"], 0, 1), interpolation="lanczos")
        ax_e.axis("off")
        ax_e.set_title(
            f"PSNR {s['pe']:.2f} dB,  SSIM {s['se']:.4f}\n"
            f"dPSNR {dp:+.2f} dB,  LPIPS {s['lp']:.4f}",
            fontsize=9, pad=4, color=col)
        if k == 0:
            ax_e.text(-0.08, 0.5, "Enhanced", transform=ax_e.transAxes,
                      ha="right", va="center", rotation=90,
                      fontsize=11, fontweight="bold", color="#0D6E6E")
        fn = (s["fname"][:18] + "..") if len(s["fname"]) > 18 else s["fname"]
        ax_e.text(0.5, -0.04, fn, transform=ax_e.transAxes,
                  ha="center", fontsize=8, color="#aaa")

    # Footer
    ax_f = fig.add_subplot(gs[2])
    ax_f.set_xlim(0, 1)
    ax_f.set_ylim(0, 1)
    ax_f.axis("off")
    txt = "\n".join([
        f"Dataset: {label}     N = {result['n']}",
        f"PSNR:  {result['psnr_inp']:.2f} dB  ->  {result['psnr_enh']:.2f} dB   (dPSNR {result['dpsnr']:+.2f} dB)",
        f"SSIM:  {result['ssim_inp']:.4f}  ->  {result['ssim_enh']:.4f}",
        f"LPIPS: {result['lpips']:.4f}     H: {result['mean_H']:.4f}     Dominant: {result['dom_filter']}",
    ])
    ax_f.text(0.01, 0.97, txt, transform=ax_f.transAxes, va="top", ha="left",
              fontsize=9, color="#111", linespacing=1.6,
              bbox=dict(boxstyle="round,pad=0.45", fc="#f8f8f8", ec="#ccc", lw=0.7))

    fw   = result["mean_fw"]
    x0   = 0.60
    xmax = 0.97
    ys   = np.linspace(0.88, 0.10, 5)
    bh   = 0.095
    ax_f.text(x0, 0.97, "Mean ESS blending weights", transform=ax_f.transAxes,
              va="top", ha="left", fontsize=9, fontweight="bold", color="#333")
    for k, (fn, w, fc) in enumerate(zip(FILTER_NAMES, fw, FILTER_COLORS)):
        bw = w * (xmax - x0 - 0.12)
        ax_f.add_patch(plt.Rectangle(
            (x0+0.12, ys[k]-bh/2), xmax-x0-0.12, bh,
            transform=ax_f.transAxes, fc="#eeeeee", ec="none", zorder=1))
        ax_f.add_patch(plt.Rectangle(
            (x0+0.12, ys[k]-bh/2), bw, bh,
            transform=ax_f.transAxes, fc=fc, ec="none", alpha=0.88, zorder=2))
        ax_f.text(x0+0.11, ys[k], fn, transform=ax_f.transAxes,
                  va="center", ha="right", fontsize=8.5, color="#333")
        ax_f.text(x0+0.12+bw+0.012, ys[k], f"{w*100:.1f}%",
                  transform=ax_f.transAxes,
                  va="center", ha="left", fontsize=8.5, color="#333")

    fig.text(0.5, 0.975, f"Figure: EAIM-Net v5 — {label} Enhancement Results",
             ha="center", va="top", fontsize=12, fontweight="bold", color="#1B3A6B")

    safe    = label.replace(" ","_").replace("/","_").replace("(","").replace(")","")
    out_png = os.path.join(EVAL_OUT, f"Fig_{safe}.png")
    out_jpg = os.path.join(EVAL_OUT, f"Fig_{safe}.jpg")
    fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight",
                facecolor="white", format="png")
    fig.savefig(out_jpg, dpi=SAVE_DPI, bbox_inches="tight",
                facecolor="white", format="jpeg",
                pil_kwargs={"quality": 97, "subsampling": 0})
    plt.show()
    print(f"  PNG {SAVE_DPI} dpi: {os.path.basename(out_png)}"
          f"  ({os.path.getsize(out_png)/1e6:.1f} MB)")
    print(f"  JPG {SAVE_DPI} dpi: {os.path.basename(out_jpg)}"
          f"  ({os.path.getsize(out_jpg)/1e6:.1f} MB)")
    plt.close()


def eval_dataset(pairs, label, n_vis=4, max_eval=None):
    """
    Evaluate MODEL on (inp_path, gt_path) pairs.
    Prints first 3 errors per run so failures are visible.
    """
    if not pairs:
        print(f"  {label}: no pairs — skipping")
        return None

    random.shuffle(pairs)
    if max_eval:
        pairs = pairs[:max_eval]

    psnrs_i=[]; ssims_i=[]
    psnrs_e=[]; ssims_e=[]; lpips_e=[]
    fw_all=[]; H_all=[]; samples=[]
    n_err = 0

    print(f"\n{'='*56}")
    print(f"  Evaluating: {label}  ({len(pairs)} pairs)")
    print(f"{'='*56}")

    for idx, (ip, gp) in enumerate(tqdm(pairs, desc=f"  {label}", leave=False)):
        try:
            inp_pil = Image.open(ip).convert("RGB")
            gt_pil  = Image.open(gp).convert("RGB")
            inp_np  = np.array(inp_pil).astype(np.float32) / 255.0
            gt_np   = np.array(gt_pil ).astype(np.float32) / 255.0
            gt_np   = _resize_match(inp_np, gt_np)

            enh_np, fw, H, wl = _run_model(MODEL, inp_pil)

            pi = psnr_fn(gt_np, inp_np, data_range=1.0)
            si = ssim_fn(gt_np, inp_np, data_range=1.0, channel_axis=2)
            pe, se, lp = _metrics(enh_np, gt_np)

            psnrs_i.append(pi); ssims_i.append(si)
            psnrs_e.append(pe); ssims_e.append(se)
            lpips_e.append(lp); fw_all.append(fw); H_all.append(H)

            if idx < n_vis:
                samples.append(dict(
                    inp_np=inp_np, enh_np=enh_np,
                    fname=os.path.basename(ip),
                    fw=fw, H=H, wl=wl,
                    pi=pi, si=si, pe=pe, se=se, lp=lp
                ))

        except Exception as err:
            n_err += 1
            if n_err <= 3:
                print(f"  [skip {idx}] {os.path.basename(ip)}: "
                      f"{type(err).__name__}: {err}")
            elif n_err == 4:
                print("  (further errors suppressed...)")

    if not psnrs_e:
        print(f"  ERROR: 0/{len(pairs)} pairs succeeded.")
        print(f"  All {n_err} pairs failed — see error messages above.")
        return None

    if n_err:
        print(f"  Processed: {len(psnrs_e)}/{len(pairs)}  ({n_err} skipped)")

    mean_fw = np.mean(fw_all, axis=0)
    result = dict(
        n          = len(psnrs_e),
        psnr_inp   = round(float(np.mean(psnrs_i)), 2),
        psnr_enh   = round(float(np.mean(psnrs_e)), 2),
        ssim_inp   = round(float(np.mean(ssims_i)), 4),
        ssim_enh   = round(float(np.mean(ssims_e)), 4),
        lpips      = round(float(np.mean(lpips_e)), 4),
        dpsnr      = round(float(np.mean(psnrs_e)) - float(np.mean(psnrs_i)), 2),
        mean_fw    = mean_fw.tolist(),
        mean_H     = round(float(np.mean(H_all)), 4),
        dom_filter = FILTER_NAMES[int(np.argmax(mean_fw))],
        samples    = samples,
    )
    RESULTS[label] = result

    print(f"  PSNR  : {result['psnr_inp']:.2f} -> {result['psnr_enh']:.2f} dB"
          f"  (dPSNR {result['dpsnr']:+.2f})")
    print(f"  SSIM  : {result['ssim_inp']:.4f} -> {result['ssim_enh']:.4f}")
    print(f"  LPIPS :                    {result['lpips']:.4f}")
    print(f"  H     : {result['mean_H']:.4f}   dominant: {result['dom_filter']}")

    make_figure(label, samples[:n_vis], result)
    return result


# ── Pair finder helpers ───────────────────────────────────────
def _find_pairs_lol(root):
    pairs = []; exts = (".jpg",".png",".jpeg")
    for r, dirs, _ in os.walk(root):
        dc = {d.lower(): d for d in dirs}
        if "low" in dc and "high" in dc:
            ld = os.path.join(r, dc["low"])
            hd = os.path.join(r, dc["high"])
            hlut = {os.path.splitext(f)[0]: os.path.join(hd, f)
                    for f in os.listdir(hd) if f.lower().endswith(exts)}
            for f in sorted(os.listdir(ld)):
                if not f.lower().endswith(exts): continue
                st = os.path.splitext(f)[0]
                if st in hlut:
                    pairs.append((os.path.join(ld, f), hlut[st]))
    return pairs


def _find_pairs_reside(hazy_d, clear_d):
    exts = (".jpg",".png",".jpeg")
    clut = {os.path.splitext(f)[0]: os.path.join(clear_d, f)
            for f in os.listdir(clear_d) if f.lower().endswith(exts)}
    pairs = []
    for f in sorted(os.listdir(hazy_d)):
        if not f.lower().endswith(exts): continue
        st = os.path.splitext(f)[0].split("_")[0]
        if st in clut:
            pairs.append((os.path.join(hazy_d, f), clut[st]))
    return pairs


def _find_pairs_rain(rain_d, clean_d):
    import re as _re
    exts = (".jpg",".png",".jpeg")
    clut = {os.path.splitext(f)[0]: os.path.join(clean_d, f)
            for f in os.listdir(clean_d) if f.lower().endswith(exts)}
    pairs = []
    for f in sorted(os.listdir(rain_d)):
        if not f.lower().endswith(exts): continue
        st = os.path.splitext(f)[0]
        gt = (clut.get(st) or
              clut.get(st[:-2] if st.endswith("x2") else "") or
              clut.get(_re.sub(r"x[0-9]+$", "", st)))
        if gt:
            pairs.append((os.path.join(rain_d, f), gt))
    return pairs


def _find_rain_dirs(root):
    for r, dirs, _ in os.walk(root):
        if r[len(root):].count(os.sep) > 3: continue
        dc = {d.lower(): d for d in dirs}
        if "rain" in dc:
            for cn in ["norain","clean","clear","gt"]:
                if cn in dc:
                    return os.path.join(r,dc["rain"]), os.path.join(r,dc[cn])
    return None, None


def _find_reside_dirs(root):
    for r, dirs, _ in os.walk(root):
        if r[len(root):].count(os.sep) > 3: continue
        dc = {d.lower(): d for d in dirs}
        if "hazy" in dc and "clear" in dc:
            return os.path.join(r,dc["hazy"]), os.path.join(r,dc["clear"])
    return None, None


print("Evaluation engine ready.")
print("  400 dpi PNG + JPG  |  Times New Roman  |  Errors printed per image")


---
## Cell 4a — LOL eval15 (low-light)
*Extracts LOL.zip only → evaluates → deletes.*

In [ ]:
# ================================================================
# CELL 4a — LOL eval15  (low-light)
# Extracts LOL.zip ONLY — nothing else
# ================================================================
import shutil, gc

MAX_EVAL = None  # all 15 eval15 pairs
N_VIS    = 4

# ── Step 1: Extract LOL.zip only ─────────────────────────────
print("Step 1: Extracting LOL.zip only...")
_lol_root = _extract_one_zip(
    zip_name_patterns=["LOL*.zip","lol*.zip"],
    extract_to="/content/datasets",
    expected_subdir="LOL"
)
print(f"  Root: {_lol_root}")

# ── Step 2: Find eval15 pairs ────────────────────────────────
print("\nStep 2: Finding LOL eval15 pairs...")
_pairs = []
if _lol_root and os.path.isdir(_lol_root):
    # LOL structure: LOL/train/ and LOL/eval/
    # Walk to find eval/low + eval/high
    for _r,_dirs,_ in os.walk(_lol_root):
        _dc = {d.lower():d for d in _dirs}
        if "eval" in _dc or "eval15" in _dc:
            _sub = _dc.get("eval") or _dc.get("eval15")
            _ep  = os.path.join(_r, _sub)
            _ec  = {d.lower():d for d in os.listdir(_ep)}
            if "low" in _ec and "high" in _ec:
                _ld = os.path.join(_ep, _ec["low"])
                _hd = os.path.join(_ep, _ec["high"])
                _hlut = {os.path.splitext(f)[0]: os.path.join(_hd,f)
                         for f in os.listdir(_hd)
                         if f.lower().endswith((".jpg",".png",".jpeg"))}
                for _f in sorted(os.listdir(_ld)):
                    if not _f.lower().endswith((".jpg",".png",".jpeg")): continue
                    _st = os.path.splitext(_f)[0]
                    if _st in _hlut:
                        _pairs.append((os.path.join(_ld,_f), _hlut[_st]))
            if _pairs: break
    if not _pairs:
        print("  eval15 not found, trying train split...")
        _pairs = _find_pairs_lol(_lol_root)

print(f"  Found: {len(_pairs)} pairs")
if _pairs:
    print(f"  Input : {_pairs[0][0]}")
    print(f"  Target: {_pairs[0][1]}")

# ── Step 3: Evaluate ─────────────────────────────────────────
result_lol = None
if _pairs:
    print(f"\nStep 3: Evaluating {len(_pairs)} pairs...")
    result_lol = eval_dataset(_pairs, "LOL eval15", n_vis=N_VIS, max_eval=MAX_EVAL)
else:
    print("ERROR: No pairs found. Check LOL.zip is in Weather folder.")

# ── Step 4: Cleanup ──────────────────────────────────────────
print("\nStep 4: Cleanup...")
shutil.rmtree("/content/datasets/LOL", ignore_errors=True)
gc.collect()
print("  /content/datasets/LOL deleted. Disk free.")


---
## Cell 4b — RESIDE-ITS (haze)
*Extracts RESIDE zip only → evaluates → deletes.*

In [ ]:
# ================================================================
# CELL 4b — RESIDE-ITS  (haze)
# Extracts RESIDE zip ONLY
# ================================================================
import shutil, gc

MAX_EVAL = 200
N_VIS    = 4

# ── Step 1: Extract ──────────────────────────────────────────
print("Step 1: Extracting RESIDE zip only...")
_reside_root = _extract_one_zip(
    zip_name_patterns=["RESIDE*.zip","reside*.zip","RESIDE_SOTS*.zip"],
    extract_to="/content/datasets",
    expected_subdir=None
)
print(f"  Root: {_reside_root}")

# ── Step 2: Find pairs ───────────────────────────────────────
print("\nStep 2: Finding RESIDE pairs...")
_pairs = []
if _reside_root and os.path.isdir(_reside_root):
    _hazy, _clear = _find_reside_dirs(_reside_root)
    print(f"  hazy : {_hazy}")
    print(f"  clear: {_clear}")
    if _hazy and _clear:
        _pairs = _find_pairs_reside(_hazy, _clear)
print(f"  Found: {len(_pairs)} pairs")

# ── Step 3: Evaluate ─────────────────────────────────────────
result_reside = None
if _pairs:
    print(f"\nStep 3: Evaluating (up to {MAX_EVAL}) pairs...")
    result_reside = eval_dataset(_pairs, "RESIDE-ITS", n_vis=N_VIS, max_eval=MAX_EVAL)
else:
    print("ERROR: No pairs found.")

# ── Step 4: Cleanup ──────────────────────────────────────────
print("\nStep 4: Cleanup...")
for _d in [_reside_root,"/content/datasets/RESIDE_SOTS","/content/datasets/RESIDE"]:
    if _d and os.path.isdir(_d):
        shutil.rmtree(_d, ignore_errors=True)
        print(f"  Deleted: {_d}")
gc.collect()
print("  Disk free.")


---
## Cell 4c — Rain100L (light rain)
*Extracts Rain100L.zip only → evaluates → deletes.*

In [ ]:
# ================================================================
# CELL 4c — Rain100L  (light rain)
# Extracts Rain100L.zip ONLY
# ================================================================
import shutil, gc

MAX_EVAL = None  # all 100 test pairs
N_VIS    = 4

# ── Step 1: Extract ──────────────────────────────────────────
print("Step 1: Extracting Rain100L.zip only...")
_r100l_root = _extract_one_zip(
    zip_name_patterns=["Rain100L*.zip","rain100l*.zip","Rain100L.zip"],
    extract_to="/content/datasets",
    expected_subdir="Rain100L"
)
print(f"  Root: {_r100l_root}")

# ── Step 2: Find pairs ───────────────────────────────────────
print("\nStep 2: Finding pairs...")
_pairs = []
if _r100l_root and os.path.isdir(_r100l_root):
    _rd, _cd = _find_rain_dirs(_r100l_root)
    print(f"  rain : {_rd}")
    print(f"  clean: {_cd}")
    if _rd and _cd:
        _pairs = _find_pairs_rain(_rd, _cd)
print(f"  Found: {len(_pairs)} pairs")
if _pairs:
    print(f"  Sample: {os.path.basename(_pairs[0][0])} -> {os.path.basename(_pairs[0][1])}")

# ── Step 3: Evaluate ─────────────────────────────────────────
result_r100l = None
if _pairs:
    print(f"\nStep 3: Evaluating {len(_pairs)} pairs...")
    result_r100l = eval_dataset(_pairs, "Rain100L", n_vis=N_VIS, max_eval=MAX_EVAL)
else:
    print("ERROR: No pairs found.")

# ── Step 4: Cleanup ──────────────────────────────────────────
print("\nStep 4: Cleanup...")
shutil.rmtree("/content/datasets/Rain100L", ignore_errors=True)
gc.collect()
print("  Disk free.")


---
## Cell 4d — Rain100H (heavy rain)
*Extracts Rain100H.zip only → evaluates → deletes.*

In [ ]:
# ================================================================
# CELL 4d — Rain100H  (heavy rain)
# Extracts Rain100H.zip ONLY
# ================================================================
import shutil, gc

MAX_EVAL = 100
N_VIS    = 4

# ── Step 1: Extract ──────────────────────────────────────────
print("Step 1: Extracting Rain100H.zip only...")
_r100h_root = _extract_one_zip(
    zip_name_patterns=["Rain100H*.zip","rain100h*.zip","Rain100H.zip"],
    extract_to="/content/datasets",
    expected_subdir="Rain100H"
)
print(f"  Root: {_r100h_root}")

# ── Step 2: Find pairs ───────────────────────────────────────
print("\nStep 2: Finding pairs...")
_pairs = []
if _r100h_root and os.path.isdir(_r100h_root):
    _rd, _cd = _find_rain_dirs(_r100h_root)
    print(f"  rain : {_rd}")
    print(f"  clean: {_cd}")
    if _rd and _cd:
        _pairs = _find_pairs_rain(_rd, _cd)
print(f"  Found: {len(_pairs)} pairs")

# ── Step 3: Evaluate ─────────────────────────────────────────
result_r100h = None
if _pairs:
    print(f"\nStep 3: Evaluating (up to {MAX_EVAL}) pairs...")
    result_r100h = eval_dataset(_pairs, "Rain100H", n_vis=N_VIS, max_eval=MAX_EVAL)
else:
    print("ERROR: No pairs found.")

# ── Step 4: Cleanup ──────────────────────────────────────────
print("\nStep 4: Cleanup...")
shutil.rmtree("/content/datasets/Rain100H", ignore_errors=True)
gc.collect()
print("  Disk free.")


---
## Cell 4e — DID-MDN (medium rain)
*Extracts DID-MDN.zip only → evaluates test split → deletes.*

In [ ]:
# ================================================================
# CELL 4e — DID-MDN  (medium rain)
# Extracts DID-MDN.zip ONLY
# ================================================================
import shutil, gc

MAX_EVAL = 200
N_VIS    = 4

# ── Step 1: Extract ──────────────────────────────────────────
print("Step 1: Extracting DID-MDN.zip only...")
_did_root = _extract_one_zip(
    zip_name_patterns=["DID*.zip","did*.zip","DID-MDN*.zip","medium_dataset*.zip"],
    extract_to="/content/datasets",
    expected_subdir=None
)
print(f"  Root: {_did_root}")

# ── Step 2: Find pairs ───────────────────────────────────────
print("\nStep 2: Finding pairs...")
_pairs = []
if _did_root and os.path.isdir(_did_root):
    for _sp in ["test","Test"]:
        _rd = os.path.join(_did_root, _sp, "rain")
        for _cn in ["clean","clear"]:
            _cd = os.path.join(_did_root, _sp, _cn)
            if os.path.isdir(_rd) and os.path.isdir(_cd):
                _pairs = _find_pairs_rain(_rd, _cd)
                print(f"  rain : {_rd}")
                print(f"  clean: {_cd}")
                break
        if _pairs: break
print(f"  Found: {len(_pairs)} pairs")

# ── Step 3: Evaluate ─────────────────────────────────────────
result_did = None
if _pairs:
    print(f"\nStep 3: Evaluating (up to {MAX_EVAL}) pairs...")
    result_did = eval_dataset(_pairs, "DID-MDN", n_vis=N_VIS, max_eval=MAX_EVAL)
else:
    print("ERROR: No pairs found. Check DID-MDN zip structure.")

# ── Step 4: Cleanup ──────────────────────────────────────────
print("\nStep 4: Cleanup...")
if _did_root and os.path.isdir(_did_root):
    shutil.rmtree(_did_root, ignore_errors=True)
    print(f"  Deleted: {_did_root}")
gc.collect()
print("  Disk free.")


---
## Cell 4f — SD1 (glare)
*Extracts sd1.zip only → evaluates → deletes.*

In [ ]:
# ================================================================
# CELL 4f — SD1  (glare)
# Extracts sd1.zip ONLY
# ================================================================
import shutil, gc, tempfile as _tf

MAX_EVAL = 200
N_VIS    = 4

# ── Step 1: Extract ──────────────────────────────────────────
print("Step 1: Extracting SD1 zip only...")
_sd1_root = _extract_one_zip(
    zip_name_patterns=["[Ss][Dd]1*.zip","[Ss][Dd]1.zip","sd1.zip","SD1.zip"],
    extract_to="/content/datasets",
    expected_subdir=None
)
print(f"  Root: {_sd1_root}")

# ── Step 2: Find pairs ───────────────────────────────────────
print("\nStep 2: Finding pairs...")
_pairs = []
exts = (".jpg",".png",".jpeg",".bmp")

if _sd1_root and os.path.isdir(_sd1_root):
    # Try *_gt + *_light test pairs first
    for _sub in ["test","Test","val","Val",""]:
        _d = os.path.join(_sd1_root,_sub) if _sub else _sd1_root
        if not os.path.isdir(_d): continue
        for _gf in [f for f in os.listdir(_d)
                    if "_gt." in f.lower() and f.lower().endswith(exts)]:
            _stem = _gf.lower().split("_gt")[0]
            for _ext in exts:
                _lf = os.path.join(_d, _stem+"_light"+_ext)
                if os.path.exists(_lf):
                    _pairs.append((_lf, os.path.join(_d,_gf)))
                    break
        if _pairs:
            print(f"  Using test pairs from: {_d}")
            break

    # Fallback: 3-panel strips
    if not _pairs:
        print("  No *_gt/*_light pairs found, trying 3-panel strips...")
        _strips = []
        for _root2,_,_files in os.walk(_sd1_root):
            for _sf in _files:
                if _sf.lower().endswith(exts):
                    _strips.append(os.path.join(_root2,_sf))
            if len(_strips)>=MAX_EVAL: break

        print(f"  Found {len(_strips)} strip files")
        _tmp_dir = _tf.mkdtemp()
        for _sidx, _sp in enumerate(_strips[:MAX_EVAL]):
            try:
                _img = Image.open(_sp).convert("RGB")
                _W,_H = _img.size; _pw=_W//3
                _ti = os.path.join(_tmp_dir, f"inp_{_sidx:04d}.jpg")
                _tg = os.path.join(_tmp_dir, f"gt_{_sidx:04d}.jpg")
                _img.crop((_pw,0,_pw*2,_H)).save(_ti)
                _img.crop((0,0,_pw,_H)).save(_tg)
                _pairs.append((_ti,_tg))
            except: pass

print(f"  Total pairs: {len(_pairs)}")

# ── Step 3: Evaluate ─────────────────────────────────────────
result_sd1 = None
if _pairs:
    print(f"\nStep 3: Evaluating (up to {MAX_EVAL}) pairs...")
    result_sd1 = eval_dataset(_pairs, "SD1 glare", n_vis=N_VIS, max_eval=MAX_EVAL)
else:
    print("ERROR: No SD1 pairs found.")

# ── Step 4: Cleanup ──────────────────────────────────────────
print("\nStep 4: Cleanup...")
if _sd1_root and os.path.isdir(_sd1_root):
    shutil.rmtree(_sd1_root, ignore_errors=True)
    print(f"  Deleted: {_sd1_root}")
try:
    shutil.rmtree(_tmp_dir, ignore_errors=True)
    print(f"  Deleted temp: {_tmp_dir}")
except: pass
gc.collect()
print("  Disk free.")


---
## Cell 4g — WTT Synthetic (all conditions)
*Extracts weather_time_data.zip only → evaluates test split → deletes.*

In [ ]:
# ================================================================
# CELL 4g — WTT Synthetic  (full pipeline, all conditions)
# Extracts weather_time_data.zip ONLY
# ================================================================
import shutil, gc

MAX_EVAL = 200
N_VIS    = 4

# ── Step 1: Extract ──────────────────────────────────────────
print("Step 1: Extracting weather_time_data.zip only...")
_wtt_root = _extract_one_zip(
    zip_name_patterns=["weather_time_data*.zip","WTT*.zip","wtt*.zip"],
    extract_to="/content/datasets",
    expected_subdir="weather_time_data"
)
print(f"  Root: {_wtt_root}")

# ── Step 2: Find pairs ───────────────────────────────────────
print("\nStep 2: Finding WTT test pairs...")
_pairs = []
exts = (".jpg",".png",".jpeg")
if _wtt_root and os.path.isdir(_wtt_root):
    _td = os.path.join(_wtt_root,"test","images")
    _cd = os.path.join(_wtt_root,"test","clean_images")
    print(f"  images : {_td}  exists={os.path.isdir(_td)}")
    print(f"  clean  : {_cd}  exists={os.path.isdir(_cd)}")
    if os.path.isdir(_td) and os.path.isdir(_cd):
        for _f in sorted(os.listdir(_td)):
            if not _f.lower().endswith(exts): continue
            _cp = os.path.join(_cd,_f)
            if os.path.exists(_cp):
                _pairs.append((os.path.join(_td,_f),_cp))
print(f"  Found: {len(_pairs)} pairs")

# ── Step 3: Evaluate ─────────────────────────────────────────
result_wtt = None
if _pairs:
    print(f"\nStep 3: Evaluating (up to {MAX_EVAL}) pairs...")
    result_wtt = eval_dataset(_pairs, "WTT Synthetic", n_vis=N_VIS, max_eval=MAX_EVAL)
else:
    print("ERROR: No WTT pairs found.")

# ── Step 4: Cleanup ──────────────────────────────────────────
print("\nStep 4: Cleanup...")
for _d in ["/content/datasets/WTT","/content/datasets/weather_time_data",_wtt_root]:
    if _d and os.path.isdir(_d):
        shutil.rmtree(_d, ignore_errors=True)
        print(f"  Deleted: {_d}")
gc.collect()
print("  Disk free.")


---
## Cell 5 — Summary Table, CSV, LaTeX, Chart
*Run after all Cell 4x complete.*

---
## Cell 6a — Fig 6: Training convergence curves
*Reads training_log_v5.txt from Drive. Generates 4-panel convergence figure at 400 dpi.*

In [ ]:
# ================================================================
# CELL — Fig 6: Training convergence curves (400 dpi, Times New Roman)
# Reads training_log_v5.txt from Drive and generates publication figure
# ================================================================
import matplotlib as _mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as GS
import numpy as np, os, re

_mpl.rcParams.update({
    "font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],
    "font.size":11,"axes.titlesize":11,"axes.labelsize":11,
    "xtick.labelsize":10,"ytick.labelsize":10,
})
SAVE_DPI = 400

# ── Read training log ─────────────────────────────────────────
LOG_PATH = os.path.join(DRIVE_ROOT, "training_log_v5.txt")
assert os.path.exists(LOG_PATH), f"Log not found: {LOG_PATH}"

epochs=[]; train_enh=[]; val_enh=[]; H_vals=[]; W_vals=[]; T_vals=[]; tau_vals=[]

with open(LOG_PATH) as f:
    for line in f:
        # Match lines like: 31   B  epe=0.19  enh=0.149  H=0.69  val=0.149  W=0.972  t=0.988
        m = re.match(
            r"\s*(\d+)\s+[AB]\s+epe=([\d.]+)\s+enh=([\d.]+)\s+H=([\d.]+)"
            r"\s+val=([\d.]+)\s+W=([\d.]+)\s+t=([\d.]+)", line)
        if m:
            epochs.append(int(m.group(1)))
            train_enh.append(float(m.group(3)))
            H_vals.append(float(m.group(4)))
            val_enh.append(float(m.group(5)))
            W_vals.append(float(m.group(6)))
            T_vals.append(float(m.group(7)))
            tau_vals.append(float(m.group(7)))

print(f"Parsed {len(epochs)} training epochs from log")
print(f"  Epoch range: {min(epochs)} -- {max(epochs)}")

# ── Build figure ──────────────────────────────────────────────
fig = plt.figure(figsize=(12, 8), facecolor="white")
gs  = GS.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# Phase B boundary (epoch 10)
def shade_phases(ax, ep_list):
    if len(ep_list) < 2: return
    b10 = 10
    ax.axvspan(min(ep_list), b10, alpha=0.06, color="#378ADD", label="Phase A")
    ax.axvspan(b10, max(ep_list), alpha=0.05, color="#1D9E75", label="Phase B")
    ax.axvline(b10, color="#888", lw=0.8, ls="--", alpha=0.5)

# (a) Enhancement loss
ax1 = fig.add_subplot(gs[0,0])
ax1.plot(epochs, train_enh, color="#378ADD", lw=1.5, ls="--", label="Train enh loss")
ax1.plot(epochs, val_enh,   color="#D85A30", lw=2.0,           label="Val enh loss")
shade_phases(ax1, epochs)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Enhancement loss")
ax1.set_title("(a) Enhancement loss")
ax1.legend(fontsize=9); ax1.grid(alpha=0.3); ax1.set_axisbelow(True)

# (b) Weight entropy H
ax2 = fig.add_subplot(gs[0,1])
ax2.plot(epochs, H_vals, color="#1D9E75", lw=2.0, label="Entropy H")
ax2.axhline(0.20, color="#888", lw=0.8, ls=":", label="H>0.20 threshold")
shade_phases(ax2, epochs)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Weight entropy H")
ax2.set_title("(b) Weight entropy H")
ax2.set_ylim(0, 1); ax2.legend(fontsize=9); ax2.grid(alpha=0.3); ax2.set_axisbelow(True)

# (c) Accuracy
ax3 = fig.add_subplot(gs[1,0])
ax3.plot(epochs, [w*100 for w in W_vals], color="#7F77DD", lw=1.5, label="Weather acc.")
ax3.plot(epochs, [t*100 for t in T_vals], color="#EF9F27", lw=1.5, ls="--", label="Time acc.")
shade_phases(ax3, epochs)
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Accuracy (%)")
ax3.set_title("(c) EPE classification accuracy")
ax3.set_ylim(50, 102); ax3.legend(fontsize=9); ax3.grid(alpha=0.3); ax3.set_axisbelow(True)

# (d) Temperature tau
ax4 = fig.add_subplot(gs[1,1])
ax4.plot(epochs, tau_vals, color="#D85A30", lw=1.5, label="Temperature tau")
ax4.axhline(0.15, color="#b31b1b", lw=0.8, ls=":", label="Collapse threshold (v4)")
shade_phases(ax4, epochs)
ax4.set_xlabel("Epoch"); ax4.set_ylabel("Temperature tau")
ax4.set_title("(d) ESS temperature tau")
ax4.set_ylim(0.5, 1.05); ax4.legend(fontsize=9); ax4.grid(alpha=0.3); ax4.set_axisbelow(True)

# Legend for phase shading
from matplotlib.patches import Patch
fig.legend(
    handles=[Patch(fc="#378ADD",alpha=0.15,label="Phase A (ESS warm-up, ep 0-10)"),
             Patch(fc="#1D9E75",alpha=0.15,label="Phase B (joint fine-tuning, ep 10+)")],
    loc="lower center", ncol=2, fontsize=9, framealpha=0.8,
    bbox_to_anchor=(0.5,-0.02)
)

fig.suptitle("Fig. 6: EAIM-Net v5 Training Convergence",
             fontsize=13, fontweight="bold", y=1.01, color="#1B3A6B")

plt.tight_layout()
out_png = os.path.join(EVAL_OUT, "Fig_6_Training_Convergence.png")
out_jpg = os.path.join(EVAL_OUT, "Fig_6_Training_Convergence.jpg")
fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
fig.savefig(out_jpg, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white",
            pil_kwargs={"quality":97})
plt.show()
print(f"PNG: {out_png}  ({os.path.getsize(out_png)/1e6:.1f} MB)")
print(f"JPG: {out_jpg}  ({os.path.getsize(out_jpg)/1e6:.1f} MB)")
plt.close()


---
## Cell 6b — Fig 7: Qualitative comparison figure
*Run after all Cell 4x complete. Generates 2-row Input/Enhanced panel for all datasets.*

In [ ]:
# ================================================================
# CELL — Fig 7: Qualitative visual comparison (Input | Enhanced)
# Run AFTER Cell 4a-4g so RESULTS dict is populated with samples
# Creates one 2-row panel per dataset then one combined figure
# ================================================================
import matplotlib as _mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as GS
import numpy as np, os

_mpl.rcParams.update({
    "font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],
    "font.size":11,"axes.titlesize":10,"axes.labelsize":10,
    "xtick.labelsize":9,"ytick.labelsize":9,
})
SAVE_DPI = 400

if not RESULTS:
    print("RESULTS is empty. Run Cells 4a-4g first.")
else:
    # ── Combined figure: 2 rows (input/enhanced) x N samples per dataset ──
    # Pick 2 best samples per dataset (highest dPSNR)
    N_PER_DS = 2
    all_samples = []   # list of (dataset_label, sample_dict)

    for ds_label, result in RESULTS.items():
        samps = sorted(result.get("samples",[]),
                       key=lambda s: s["pe"]-s["pi"], reverse=True)
        for s in samps[:N_PER_DS]:
            all_samples.append((ds_label, s))

    if not all_samples:
        print("No samples found. Check that n_vis >= 2 in eval_dataset calls.")
    else:
        n_cols = len(all_samples)
        fig = plt.figure(figsize=(n_cols * 2.5, 9), facecolor="white")
        gs  = GS.GridSpec(3, 1, figure=fig,
                          height_ratios=[4, 4, 1.8],
                          hspace=0.10, left=0.05, right=0.97,
                          top=0.94, bottom=0.02)
        gs_inp = GS.GridSpecFromSubplotSpec(1, n_cols, subplot_spec=gs[0], wspace=0.04)
        gs_enh = GS.GridSpecFromSubplotSpec(1, n_cols, subplot_spec=gs[1], wspace=0.04)

        for k, (ds_label, s) in enumerate(all_samples):
            dp  = s["pe"] - s["pi"]
            col = "#1a6b1a" if dp >= 0 else "#b31b1b"

            # Input row
            ax_i = fig.add_subplot(gs_inp[0, k])
            ax_i.imshow(np.clip(s["inp_np"], 0, 1), interpolation="lanczos")
            ax_i.axis("off")
            ax_i.set_title(f"{s['pi']:.1f} dB", fontsize=8, pad=3, color="#444")
            if k == 0:
                ax_i.text(-0.10, 0.5, "Input", transform=ax_i.transAxes,
                          ha="right", va="center", rotation=90,
                          fontsize=10, fontweight="bold", color="#1B3A6B")

            # Enhanced row
            ax_e = fig.add_subplot(gs_enh[0, k])
            ax_e.imshow(np.clip(s["enh_np"], 0, 1), interpolation="lanczos")
            ax_e.axis("off")
            ax_e.set_title(f"{s['pe']:.1f} dB  ({dp:+.1f})",
                           fontsize=8, pad=3, color=col)
            if k == 0:
                ax_e.text(-0.10, 0.5, "Enhanced", transform=ax_e.transAxes,
                          ha="right", va="center", rotation=90,
                          fontsize=10, fontweight="bold", color="#0D6E6E")

            # Dataset label below
            ax_e.text(0.5, -0.05,
                      ds_label if len(ds_label) <= 10 else ds_label[:9]+"..",
                      transform=ax_e.transAxes, ha="center",
                      fontsize=8, color="#888")

        # Footer: per-dataset metrics summary
        ax_f = fig.add_subplot(gs[2])
        ax_f.axis("off")
        ax_f.set_xlim(0, 1); ax_f.set_ylim(0, 1)

        y_pos = 0.92
        col_w = 1.0 / len(RESULTS)
        for col_idx, (ds_label, result) in enumerate(RESULTS.items()):
            xc = col_idx * col_w + col_w / 2
            dp_col = "#1a6b1a" if result["dpsnr"] >= 0 else "#b31b1b"
            ax_f.text(xc, y_pos, ds_label, ha="center", va="top",
                      fontsize=8, fontweight="bold", color="#1B3A6B",
                      transform=ax_f.transAxes)
            ax_f.text(xc, y_pos-0.22,
                      f"dPSNR {result['dpsnr']:+.2f} dB",
                      ha="center", va="top", fontsize=8, color=dp_col,
                      transform=ax_f.transAxes)
            ax_f.text(xc, y_pos-0.44,
                      f"SSIM {result['ssim_enh']:.4f}",
                      ha="center", va="top", fontsize=8, color="#333",
                      transform=ax_f.transAxes)
            ax_f.text(xc, y_pos-0.66,
                      f"LPIPS {result['lpips']:.4f}",
                      ha="center", va="top", fontsize=8, color="#333",
                      transform=ax_f.transAxes)

        fig.suptitle(
            "Fig. 7: EAIM-Net v5 — Qualitative Results Across All Degradation Types",
            fontsize=12, fontweight="bold", y=0.99, color="#1B3A6B")

        out_png = os.path.join(EVAL_OUT, "Fig_7_Qualitative_Comparison.png")
        out_jpg = os.path.join(EVAL_OUT, "Fig_7_Qualitative_Comparison.jpg")
        fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
        fig.savefig(out_jpg, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white",
                    pil_kwargs={"quality":97})
        plt.show()
        print(f"PNG: {out_png}  ({os.path.getsize(out_png)/1e6:.1f} MB)")
        print(f"JPG: {out_jpg}  ({os.path.getsize(out_jpg)/1e6:.1f} MB)")
        plt.close()

        # ── Also save one figure per dataset (same as eval cells already do) ──
        print("\nIndividual dataset figures already saved by eval cells.")
        print("Check EVAL_OUT for Fig_LOL_eval15.png, Fig_RESIDE-ITS.png etc.")


In [ ]:
# ================================================================
# CELL 5 — SUMMARY: table + CSV + LaTeX + bar chart
# ================================================================
import csv, matplotlib as _mpl2
_mpl2.rcParams.update({
    "font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],
    "font.size":11,"axes.titlesize":12,"savefig.dpi":400
})

if not RESULTS:
    print("No results yet. Run Cells 4a-4g first.")
else:
    print("\n" + "="*92)
    print("  EAIM-Net v5 — Evaluation Results")
    print("="*92)
    print(f"  {'Dataset':<22} {'N':>6} {'PSNR_in':>8} {'PSNR_enh':>9} "
          f"{'dPSNR':>7} {'SSIM_in':>8} {'SSIM_enh':>9} {'LPIPS':>7}")
    print("-"*92)
    for name,r in RESULTS.items():
        print(f"  {name:<22} {r['n']:>6} {r['psnr_inp']:>8.2f} {r['psnr_enh']:>9.2f} "
              f"{r['dpsnr']:>+7.2f} {r['ssim_inp']:>8.4f} "
              f"{r['ssim_enh']:>9.4f} {r['lpips']:>7.4f}")
    print("="*92)

    # CSV
    csv_path = os.path.join(EVAL_OUT,"metrics.csv")
    with open(csv_path,"w",newline="") as f:
        w=csv.writer(f)
        w.writerow(["Dataset","N","PSNR_in","PSNR_enh","dPSNR",
                    "SSIM_in","SSIM_enh","LPIPS","H","Dom_filter"])
        for name,r in RESULTS.items():
            w.writerow([name,r["n"],r["psnr_inp"],r["psnr_enh"],r["dpsnr"],
                        r["ssim_inp"],r["ssim_enh"],r["lpips"],r["mean_H"],r["dom_filter"]])
    print(f"CSV: {csv_path}")

    # LaTeX
    tex=["\\begin{table}[h]","\\centering",
         "\\caption{EAIM-Net v5 quantitative evaluation results.}",
         "\\label{tab:quant}","\\begin{tabular}{lrrrrrr}","\\hline",
         "Dataset & N & $\\mathrm{PSNR_{in}}$ & $\\mathrm{PSNR_{enh}}$ & "
         "$\\Delta$PSNR & $\\mathrm{SSIM_{enh}}$ & LPIPS \\\\","\\hline"]
    for name,r in RESULTS.items():
        tex.append(f"{name} & {r['n']} & {r['psnr_inp']:.2f} & {r['psnr_enh']:.2f} & "
                   f"{r['dpsnr']:+.2f} & {r['ssim_enh']:.4f} & {r['lpips']:.4f} \\\\")
    tex+=["\\hline","\\end{tabular}","\\end{table}"]
    tex_path=os.path.join(EVAL_OUT,"metrics_table.tex")
    with open(tex_path,"w") as f: f.write("\n".join(tex))
    print(f"LaTeX: {tex_path}")

    # Bar chart
    names  = list(RESULTS.keys())
    dpsnrs = [RESULTS[n]["dpsnr"]    for n in names]
    ssims  = [RESULTS[n]["ssim_enh"] for n in names]
    lpips_ = [RESULTS[n]["lpips"]    for n in names]
    colors = ["#378ADD","#1D9E75","#EF9F27","#7F77DD","#D85A30","#888780","#B31B1B"][:len(names)]

    fig,axes=plt.subplots(1,3,figsize=(15,4.5),facecolor="white")
    for ax,vals,title,ylabel,fmt,add0 in [
        (axes[0],dpsnrs,"dPSNR (dB)","PSNR improvement (dB)","{:+.2f}",True),
        (axes[1],ssims, "SSIM (enhanced)","SSIM","  {:.4f}",False),
        (axes[2],lpips_,"LPIPS (lower = better)","LPIPS","  {:.4f}",False),
    ]:
        bars=ax.bar(names,vals,color=colors,width=0.55,edgecolor="white")
        ax.set_title(title,fontsize=12,fontweight="bold")
        ax.set_ylabel(ylabel,fontsize=11)
        ax.tick_params(axis="x",rotation=35,labelsize=9)
        ax.grid(axis="y",alpha=0.3); ax.set_axisbelow(True)
        if add0: ax.axhline(0,color="#ccc",lw=0.8)
        for b,v in zip(bars,vals):
            ax.text(b.get_x()+b.get_width()/2,v+(0.05 if v>=0 else -0.12),
                    fmt.format(v),ha="center",va="bottom",fontsize=8.5)

    fig.suptitle("EAIM-Net v5 — Evaluation Metrics Across All Datasets",
                 fontsize=13,fontweight="bold",y=1.02,color="#1B3A6B")
    plt.tight_layout()
    chart_path=os.path.join(EVAL_OUT,"Fig_metrics_chart.png")
    fig.savefig(chart_path,dpi=SAVE_DPI,bbox_inches="tight",facecolor="white")
    plt.show()
    print(f"Chart: {chart_path}")

    print("\n" + "="*60)
    print(f"  All saved to: {EVAL_OUT}")
    print("="*60)
    for _fn in sorted(os.listdir(EVAL_OUT)):
        _sz=os.path.getsize(os.path.join(EVAL_OUT,_fn))/1e3
        print(f"  {_fn:<50} {_sz:>8.0f} KB")
